# Date Fruit Classification using PyTorch ANN

### What was learned & implemented in this notebook:
1. **Exploratory Data Analysis (EDA)**:
   - Analyzed morphological features of date fruits (Area, Perimeter, Axis lengths, Solidity, Eccentricity, etc.).
2. **Data Preprocessing & Encoding**:
   - Scaled numerical features using `StandardScaler`.
   - Encoded multiclass target labels (`BERHI`, `DEGLET`, `IRAQI`, `ROTANA`, `SAFAVI`, `SOGAY`, `VARY`) using `LabelEncoder`.
3. **Artificial Neural Network (ANN) Design**:
   - Built a PyTorch network with input dimension mapping to features, a hidden layer with 64 units and ReLU activations, and a final linear layer outputting 7 logits (representing class scores).
4. **Training & Multiclass Evaluation**:
   - Optimized using Cross Entropy Loss (`nn.CrossEntropyLoss`) and Adam optimizer over 100 epochs.
   - Measured final validation accuracy on testing loader, achieving **~93.3%** accuracy.

In [50]:
import pandas as pd
import numpy as np

In [51]:
df = pd.read_csv("DateFruit_Dataset.csv")

In [52]:
df.head()

,AREA,PERIMETER,MAJOR_AXIS,MINOR_AXIS,ECCENTRICITY,EQDIASQ,SOLIDITY,CONVEX_AREA,EXTENT,ASPECT_RATIO,...,KurtosisRR,KurtosisRG,KurtosisRB,EntropyRR,EntropyRG,EntropyRB,ALLdaub4RR,ALLdaub4RG,ALLdaub4RB,Class
0,422163,2378.908,837.8484,645.6693,0.6373,733.1539,0.9947,424428,0.7831,1.2976,...,3.2370,2.9574,4.2287,-59191263232,-50714214400,-39922372608,58.7255,54.9554,47.8400,BERHI
1,338136,2085.144,723.8198,595.2073,0.5690,656.1464,0.9974,339014,0.7795,1.2161,...,2.6228,2.6350,3.1704,-34233065472,-37462601728,-31477794816,50.0259,52.8168,47.8315,BERHI
2,526843,2647.394,940.7379,715.3638,0.6494,819.0222,0.9962,528876,0.7657,1.3150,...,3.7516,3.8611,4.7192,-93948354560,-74738221056,-60311207936,65.4772,59.2860,51.9378,BERHI
3,416063,2351.210,827.9804,645.2988,0.6266,727.8378,0.9948,418255,0.7759,1.2831,...,5.0401,8.6136,8.2618,-32074307584,-32060925952,-29575010304,43.3900,44.1259,41.1882,BERHI
4,347562,2160.354,763.9877,582.8359,0.6465,665.2291,0.9908,350797,0.7569,1.3108,...,2.7016,2.9761,4.4146,-39980974080,-35980042240,-25593278464,52.7743,50.9080,42.6666,BERHI


In [53]:
df.shape

(898, 35)

In [54]:
X = df.drop("Class", axis = 1)
y = df["Class"]

In [55]:
df["Class"].unique()

array(['BERHI', 'DEGLET', 'DOKOL', 'IRAQI', 'ROTANA', 'SAFAVI', 'SOGAY'],
      dtype=object)

In [56]:
from sklearn.preprocessing import StandardScaler, LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y)

In [57]:
from sklearn.model_selection import train_test_split

X_train,X_test, y_train, y_test = train_test_split(X,y,test_size = 0.2,random_state = 42)

In [58]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

**ANN**

In [59]:
import torch
import torch.nn as nn
import torch.optim as optim 
from torch.utils.data import DataLoader, TensorDataset


In [60]:
X_train_tensor = torch.tensor(X_train_scaled,dtype = torch.float32)
y_train_tensor = torch.tensor(y_train,dtype = torch.long)

X_test_tensor = torch.tensor(X_test_scaled,dtype = torch.float32)
y_test_tensor = torch.tensor(y_test,dtype = torch.long)

In [61]:
train_dataset = TensorDataset(X_train_tensor,y_train_tensor)
test_dataset = TensorDataset(X_test_tensor,y_test_tensor)

In [62]:
train_loader = DataLoader(train_dataset,batch_size = 32, shuffle = True)
test_loader = DataLoader(test_dataset,batch_size = 32)

In [63]:
class ANN(nn.Module):
    def __init__(self):
        super(ANN,self).__init__()

        self.model = nn.Sequential(
            nn.Linear(X.shape[1],64),
            nn.ReLU(),
            nn.Linear(64,64),
            nn.ReLU(),
            nn.Linear(64,7)
        )
    def forward (self, x):
        return self.model(x)

In [64]:
model = ANN()
criteria = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())


In [65]:
epochs = 100
for epoch in range(epochs):
    model.train()
    running_loss = 0.0 
    for xb, yb in train_loader:
        optimizer.zero_grad()
        outputs = model(xb)
        loss = criteria(outputs,yb)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    train_loss = running_loss/len(train_loader)
    print(f"epoch = {epoch+1}/{epochs}, loss ={train_loss} ")
    

epoch = 1/100, loss =1.7098456621170044 
epoch = 2/100, loss =1.078074390473573 
epoch = 3/100, loss =0.7358209685139034 
epoch = 4/100, loss =0.559018295744191 
epoch = 5/100, loss =0.46972826252812927 
epoch = 6/100, loss =0.3997864645460378 
epoch = 7/100, loss =0.3623126278752866 
epoch = 8/100, loss =0.31961458176374435 
epoch = 9/100, loss =0.29786351387915405 
epoch = 10/100, loss =0.27274585094140924 
epoch = 11/100, loss =0.2575516574408697 
epoch = 12/100, loss =0.23976482418568237 
epoch = 13/100, loss =0.23254390967928845 
epoch = 14/100, loss =0.21575404197463524 
epoch = 15/100, loss =0.20477164828259012 
epoch = 16/100, loss =0.19684774655362833 
epoch = 17/100, loss =0.18672526530597522 
epoch = 18/100, loss =0.18477881829375806 
epoch = 19/100, loss =0.18137118609055228 
epoch = 20/100, loss =0.16379221690737683 
epoch = 21/100, loss =0.1662569042781125 
epoch = 22/100, loss =0.16879139727224474 
epoch = 23/100, loss =0.15377090450214304 
epoch = 24/100, loss =0.151667

In [67]:
model.eval()
total = 0
correct = 0 
with torch.no_grad():
    for xb, yb in test_loader:
        outputs = model(xb)
        _,predicted = torch.max(outputs,1)# outputs are valus  and 1 is the axis it will give the value and index 
    
        correct += (predicted == yb ).sum().item()
        total += yb.size(0) # actutal saple size in each batch 

print("Acuraccy:", 100 * correct/total)


Acuraccy: 93.33333333333333
